### Import des librairies utiles pour ce projet

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import statsmodels.api as sm

### Chargement du dataset selon les csv remis lors de la mission

In [3]:
import pandas as pd

# Chemin vers le dossier contenant les trois fichiers CSV
chemin = r"C:\Users\frnar\OneDrive\Desktop\Projets OC 2.0\Projet 9"

# Chargement du fichier customers
clients = pd.read_csv(
    chemin + r"\customers.csv",
    sep=";",
    encoding="utf-8",
    low_memory=False
)

# Chargement du fichier products
produits = pd.read_csv(
    chemin + r"\products.csv",
    sep=";",
    encoding="utf-8",
    low_memory=False
)

# Chargement du fichier transactions
transactions = pd.read_csv(
    chemin + r"\Transactions.csv",
    sep=";",
    encoding="utf-8",
    low_memory=False
)

### Dimensionnement du dataset

In [6]:
import pandas as pd

def inspection(df):
    print("-" * 80)
    print("INSPECTION DU DATAFRAME")
    print("-" * 80)

    # Structure du DataFrame
    print("\nAffichage de la structure du df (colonnes, lignes)")
    print("-" * 80)
    print(f"Nombre de lignes : {df.shape[0]}")
    print(f"Nombre de colonnes : {df.shape[1]}")

    # Noms des colonnes
    print("\nNoms des colonnes :")
    print("-" * 80)
    print(df.columns)

    # Statistiques descriptives
    print("\nAffichage des principales informations statistiques des données du df :")
    print("-" * 80)
    print(df.describe(include="all").transpose().to_string())

    # Types des données
    print("\nTypes des données :")
    print("-" * 80)
    print(df.dtypes)

    # Nombre de valeurs uniques
    print("\nAffichage du nombre de valeurs uniques (afin de définir ce qui pourrait représenter une PK / FK) :")
    print("-" * 80)
    print(df.nunique(dropna=True))

    # Valeurs manquantes
    print("\nAffichage du nombre de valeurs manquantes :")
    print("-" * 80)
    print(df.isna().sum())

    # Doublons
    print("\nAffichage du nombre de doublons :")
    print("-" * 80)
    print(df.duplicated().sum())

    # Premières lignes
    print("\nAffichage des 5 premières lignes :")
    print("-" * 80)
    print(df.head().to_string())

    # Recherche d'une clé primaire potentielle
    colonnes_pk = [
        colonne for colonne in df.columns
        if df[colonne].notna().all() and df[colonne].is_unique
    ]

    print("\nVérification des clés primaires potentielles :")
    print("-" * 80)

    if colonnes_pk:
        print("Colonne(s) potentiellement unique(s) :", colonnes_pk)
    else:
        print("Aucune colonne ne peut constituer une clé primaire à elle seule.")

    print("\n" + "-" * 80)

In [9]:
inspection(clients)

--------------------------------------------------------------------------------
INSPECTION DU DATAFRAME
--------------------------------------------------------------------------------

Affichage de la structure du df (colonnes, lignes)
--------------------------------------------------------------------------------
Nombre de lignes : 8621
Nombre de colonnes : 3

Noms des colonnes :
--------------------------------------------------------------------------------
Index(['client_id', 'sex', 'birth'], dtype='object')

Affichage des principales informations statistiques des données du df :
--------------------------------------------------------------------------------
            count unique     top  freq         mean        std     min     25%     50%     75%     max
client_id    8621   8621  c_4410     1          NaN        NaN     NaN     NaN     NaN     NaN     NaN
sex          8621      2       f  4490          NaN        NaN     NaN     NaN     NaN     NaN     NaN
birth      8621.

In [8]:
# DATASET PRODUITS
inspection(produits)

--------------------------------------------------------------------------------
INSPECTION DU DATAFRAME
--------------------------------------------------------------------------------

Affichage de la structure du df (colonnes, lignes)
--------------------------------------------------------------------------------
Nombre de lignes : 3286
Nombre de colonnes : 3

Noms des colonnes :
--------------------------------------------------------------------------------
Index(['id_prod', 'price', 'categ'], dtype='object')

Affichage des principales informations statistiques des données du df :
--------------------------------------------------------------------------------
          count unique     top freq       mean        std   min   25%     50%    75%    max
id_prod    3286   3286  0_1421    1        NaN        NaN   NaN   NaN     NaN    NaN    NaN
price    3286.0    NaN     NaN  NaN  21.863597  29.849786  0.62  6.99  13.075  22.99  300.0
categ    3286.0    NaN     NaN  NaN   0.370359   

In [14]:
# DATASET TRANSACTIONS
inspection(transactions)


---------------------------------------------------------------------
INSPECTION DU DATAFRAME
---------------------------------------------------------------------

---------------------------------------------------------------------
Affichage de la structure du df (colonnes, lignes):
---------------------------------------------------------------------
nombre de lignes : 1048575
nombre de colonnes : 4

---------------------------------------------------------------------
Noms des colonnes :
---------------------------------------------------------------------
Index(['id_prod', 'date', 'session_id', 'client_id'], dtype='object')

---------------------------------------------------------------------
Affichage des principales informations statistiques des données du df :
---------------------------------------------------------------------
       id_prod                        date session_id client_id
count   687534                      687534     687534    687534
unique    3265      

### Nettoyage des données NULL dans le fichier transactions

In [15]:
transactions=transactions.dropna()
transactions.isnull().sum()

id_prod       0
date          0
session_id    0
client_id     0
dtype: int64

### Modification du format de la date du dataframe transactions

In [10]:
transactions['date'] = pd.to_datetime(transactions['date']).dt.floor('D')

In [17]:
transactions.info()
transactions.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   id_prod     687534 non-null  object        
 1   date        687534 non-null  datetime64[ns]
 2   session_id  687534 non-null  object        
 3   client_id   687534 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 32.0+ MB


,id_prod,date,session_id,client_id
0,0_1259,2021-03-01,s_1,c_329
1,0_1390,2021-03-01,s_2,c_664
2,0_1352,2021-03-01,s_3,c_580
3,0_1458,2021-03-01,s_4,c_7912
4,0_1358,2021-03-01,s_5,c_2033


In [18]:
produits['price'] = pd.to_numeric(produits['price'])
produits.head()
produits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3286 entries, 0 to 3285
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id_prod  3286 non-null   object 
 1   price    3286 non-null   float64
 2   categ    3286 non-null   int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 77.1+ KB


In [19]:
clients.isnull().sum()

client_id    0
sex          0
birth        0
dtype: int64

### Jointure des csv et création d'un unique csv df_lapage

In [20]:
df_transactions_produits = transactions.merge(produits, on='id_prod', how='left')
df_transactions_produits.head()

,id_prod,date,session_id,client_id,price,categ
0,0_1259,2021-03-01,s_1,c_329,11.99,0.0
1,0_1390,2021-03-01,s_2,c_664,19.37,0.0
2,0_1352,2021-03-01,s_3,c_580,4.50,0.0
3,0_1458,2021-03-01,s_4,c_7912,6.55,0.0
4,0_1358,2021-03-01,s_5,c_2033,16.49,0.0


In [21]:
df_lapage = df_transactions_produits.merge(clients, on='client_id', how='left')
df_lapage.head()

,id_prod,date,session_id,client_id,price,categ,sex,birth
0,0_1259,2021-03-01,s_1,c_329,11.99,0.0,f,1967.0
1,0_1390,2021-03-01,s_2,c_664,19.37,0.0,m,1960.0
2,0_1352,2021-03-01,s_3,c_580,4.50,0.0,m,1988.0
3,0_1458,2021-03-01,s_4,c_7912,6.55,0.0,f,1989.0
4,0_1358,2021-03-01,s_5,c_2033,16.49,0.0,f,1956.0


In [22]:
inspection(df_lapage)

--------------------------------------------------------------------------------
INSPECTION DU DATAFRAME
--------------------------------------------------------------------------------

Affichage de la structure du df (colonnes, lignes)
--------------------------------------------------------------------------------
Nombre de lignes : 1048575
Nombre de colonnes : 8

Noms des colonnes :
--------------------------------------------------------------------------------
Index(['id_prod', 'date', 'session_id', 'client_id', 'price', 'categ', 'sex',
       'birth'],
      dtype='object')

Affichage des principales informations statistiques des données du df :
--------------------------------------------------------------------------------
               count  unique       top    freq                           mean                  min                  25%                  50%                  75%                  max        std
id_prod       687534    3265     1_369    2340                  

### EXPORT DU DF_LAPAGE EN CSV

In [23]:
# Export du DataFrame dans le même dossier que tes fichiers CSV
df_lapage.to_csv(
    chemin + r"\df_lapage.csv",
    sep=";",
    encoding="utf-8",
    index=False
)

print("Export terminé : ", chemin + r"\df_lapage.csv")

Export terminé :  C:\Users\frnar\OneDrive\Desktop\Projets OC 2.0\Projet 9\df_lapage.csv
